In [6]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import urllib.parse
import plotly.graph_objects as go

# =========================
# [LOCAL] DB 접속 정보
# =========================
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres",
    "password": "",
}

def get_engine(cfg=DB_CONFIG):
    user = cfg["user"]
    password = urllib.parse.quote_plus(cfg["password"])
    host = cfg["host"]
    port = cfg["port"]
    dbname = cfg["dbname"]
    conn_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
    return create_engine(conn_str, pool_pre_ping=True)

engine = get_engine()

# =========================
# 소스 스키마/테이블 (요구사항 1)
# =========================
SRC_SCHEMA = "a3_vision_table"
SRC_TABLE  = "vision_table"

# =========================
# 쿼리 (요구사항 2~8,5,6)
# =========================
query = text(f"""
SELECT
    station,
    remark,
    barcode_information,
    step_description,
    result,
    end_day,
    end_time,
    run_time
FROM {SRC_SCHEMA}.{SRC_TABLE}
WHERE 1=1
  AND barcode_information LIKE 'B%%'              -- (2)
  AND station IN ('Vision1', 'Vision2')           -- (7)
  AND remark IN ('PD', 'Non-PD')                  -- (8)
  AND step_description = 'intensity8'             -- (3)(4)
  AND result <> 'FAIL'                            -- (5)
ORDER BY end_day ASC, end_time ASC                -- (6)
""")

df = pd.read_sql(query, engine)

# =========================
# month 생성 (요구사항 9)
# =========================
df["end_day"] = df["end_day"].astype(str).str.zfill(8)
df["month"] = df["end_day"].str.slice(0, 6)
df = df.sort_values(["end_day", "end_time"], ascending=True).reset_index(drop=True)

# =========================
# Boxplot 요약 생성 (요구사항 10)
# =========================
def _outlier_range_str(values: pd.Series, lower_fence: float, upper_fence: float):
    v = values.dropna().astype(float)
    if v.empty:
        return None, None

    lower_out = v[v < lower_fence]
    upper_out = v[v > upper_fence]

    lower_str = f"{lower_out.min():.2f}~{lower_fence:.2f}" if len(lower_out) > 0 else None
    upper_str = f"{upper_fence:.2f}~{upper_out.max():.2f}" if len(upper_out) > 0 else None
    return lower_str, upper_str

def _make_plotly_json(values: pd.Series, name: str):
    v = values.dropna().astype(float).to_numpy()
    fig = go.Figure()
    fig.add_trace(go.Box(y=v, name=name, boxpoints=False))
    return fig.to_json()

rows = []
group_cols = ["station", "remark", "month"]

for (station, remark, month), g in df.groupby(group_cols, dropna=False):
    rt = g["run_time"].dropna().astype(float)
    if rt.empty:
        continue

    q1 = float(rt.quantile(0.25))
    med = float(rt.quantile(0.50))
    q3 = float(rt.quantile(0.75))
    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    lower_str, upper_str = _outlier_range_str(rt, lower_fence, upper_fence)

    rt_in = rt[(rt >= lower_fence) & (rt <= upper_fence)]
    del_out_mean = float(rt_in.mean()) if len(rt_in) > 0 else np.nan

    plotly_json = _make_plotly_json(rt, name=f"{station}_{remark}_{month}")

    rows.append({
        "station": station,
        "remark": remark,
        "month": month,
        "sample_amount": int(len(rt)),                 # (10-4)
        "run_time_lower_outlier": lower_str,
        "q1": round(q1, 2),                            # (10-5)
        "median": round(med, 2),                       # (10-5)
        "q3": round(q3, 2),                            # (10-5)
        "run_time_upper_outlier": upper_str,
        "del_out_run_time_av": round(del_out_mean, 2), # (10-2)(10-5)
        "plotly_json": plotly_json                     # (10)
    })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.sort_values(["month","station","remark"], ascending=True).reset_index(drop=True)
summary_df.insert(0, "id", np.arange(1, len(summary_df) + 1))

summary_df

,id,station,remark,month,sample_amount,run_time_lower_outlier,q1,median,q3,run_time_upper_outlier,del_out_run_time_av,plotly_json
0,1,Vision1,Non-PD,202510,17174,None,1.8,1.8,1.9,2.05~2.70,1.84,"{""data"":[{""boxpoints"":false,""name"":""Vision1_No..."
1,2,Vision1,PD,202510,29528,1.50~1.65,1.8,1.8,1.9,2.05~2.70,1.84,"{""data"":[{""boxpoints"":false,""name"":""Vision1_PD..."
2,3,Vision2,Non-PD,202510,18290,None,1.8,1.8,1.9,2.05~2.70,1.85,"{""data"":[{""boxpoints"":false,""name"":""Vision2_No..."
3,4,Vision2,PD,202510,29242,1.50~1.65,1.8,1.8,1.9,2.05~5.70,1.84,"{""data"":[{""boxpoints"":false,""name"":""Vision2_PD..."


In [7]:
from sqlalchemy import text

TARGET_SCHEMA = "e2_vision_ct"
TARGET_TABLE  = "vision_run_time_ct"
UNIQUE_COLS   = ["station", "remark", "month"]

# =========================
# 1) 스키마/테이블 생성
# =========================
create_schema_sql = text(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA};")

create_table_sql = text(f"""
CREATE TABLE IF NOT EXISTS {TARGET_SCHEMA}.{TARGET_TABLE} (
    id                    INTEGER,
    station               TEXT NOT NULL,
    remark                TEXT NOT NULL,
    month                 TEXT NOT NULL,
    sample_amount         INTEGER,
    run_time_lower_outlier TEXT,
    q1                    DOUBLE PRECISION,
    median                DOUBLE PRECISION,
    q3                    DOUBLE PRECISION,
    run_time_upper_outlier TEXT,
    del_out_run_time_av   DOUBLE PRECISION,
    plotly_json           JSONB,
    updated_at            TIMESTAMPTZ DEFAULT now(),
    PRIMARY KEY (station, remark, month)
);
""")

with engine.begin() as conn:
    conn.execute(create_schema_sql)
    conn.execute(create_table_sql)

print(f"[OK] ensured {TARGET_SCHEMA}.{TARGET_TABLE}")

# =========================
# 2) UPSERT (station+remark+month 동일이면 덮어쓰기)
# =========================
# plotly_json 컬럼: 문자열(json) -> jsonb로 캐스팅되도록 ::jsonb 처리
upsert_sql = text(f"""
INSERT INTO {TARGET_SCHEMA}.{TARGET_TABLE} (
    id, station, remark, month,
    sample_amount, run_time_lower_outlier, q1, median, q3,
    run_time_upper_outlier, del_out_run_time_av, plotly_json, updated_at
)
VALUES (
    :id, :station, :remark, :month,
    :sample_amount, :run_time_lower_outlier, :q1, :median, :q3,
    :run_time_upper_outlier, :del_out_run_time_av, (:plotly_json)::jsonb, now()
)
ON CONFLICT (station, remark, month)
DO UPDATE SET
    id = EXCLUDED.id,
    sample_amount = EXCLUDED.sample_amount,
    run_time_lower_outlier = EXCLUDED.run_time_lower_outlier,
    q1 = EXCLUDED.q1,
    median = EXCLUDED.median,
    q3 = EXCLUDED.q3,
    run_time_upper_outlier = EXCLUDED.run_time_upper_outlier,
    del_out_run_time_av = EXCLUDED.del_out_run_time_av,
    plotly_json = EXCLUDED.plotly_json,
    updated_at = now();
""")

records = summary_df.to_dict(orient="records")

with engine.begin() as conn:
    conn.execute(upsert_sql, records)

print(f"[OK] upserted {len(records)} rows into {TARGET_SCHEMA}.{TARGET_TABLE}")

[OK] ensured e2_vision_ct.vision_run_time_ct
[OK] upserted 4 rows into e2_vision_ct.vision_run_time_ct
